# 07 位置编码 Positional Encoding

前面我们已经学到 Multi-Head Attention。

现在有一个问题必须补上：

```text
Self-Attention 让每个 token 都能看所有 token。
但模型怎么知道谁在前、谁在后？
```

这就是位置编码要解决的问题。

这一节先不写代码，只讲清楚：

```text
为什么需要位置编码
位置编码加在哪里
为什么通常是和 token embedding 相加
可学习位置编码是什么
正弦余弦位置编码是什么
位置编码和 Attention、Transformer 的关系是什么
```

## 1. 为什么学完 Multi-Head 后要学位置编码

Multi-Head Attention 解决的是：

```text
用多个关系视角，让 token 之间互相传递信息。
```

但它还有一个天然问题。

Attention 主要看的是内容之间的匹配关系。

它会计算：

```text
这个 token 和那个 token 有多相关？
```

但如果我们不给它额外信息，它不天然知道：

```text
这个 token 是第 1 个
那个 token 是第 2 个
另一个 token 是第 10 个
```

所以学完 Attention 后，下一步一定要补位置编码。

## 2. 顺序为什么重要

看两句话：

```text
我 喜欢 你
你 喜欢 我
```

这两句话用到的 token 很像。

但是意思不一样。

再看一个例子：

```text
小明 打 小红
小红 打 小明
```

同样的词，顺序换了，主语和宾语就换了。

所以文本不只是一个词袋。

它是一条有顺序的序列。

模型如果不知道位置，就很难理解这种顺序差异。

## 3. Self-Attention 本身为什么不天然知道顺序

Self-Attention 的核心计算是：

$$
\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

它关心的是 Query 和 Key 的匹配程度。

换句话说，它关心的是：

```text
谁和谁内容相关？
```

但如果输入里没有位置相关的信息，那么 Attention 看到的只是“一组 token 向量”。

它不知道这些向量原本排在第几个。

可以粗略理解成：

```text
Attention 擅长建模关系。
但位置信息需要额外告诉它。
```

## 4. 一个重要直觉：位置编码就是给 token 发座位号

可以把一句话想成一排座位。

每个 token 坐在一个位置上。

token embedding 告诉模型：

```text
这个位置坐的是谁。
```

位置编码告诉模型：

```text
这个 token 坐在第几个位置。
```

所以输入 Transformer 前，模型需要同时知道两件事：

```text
内容信息：这个 token 是什么。
位置信息：这个 token 在哪里。
```

位置编码不是替代 token embedding。

它是给 token embedding 补上顺序信息。

## 5. 位置编码加在哪里

Transformer 常见做法是：

```text
输入向量 = token embedding + positional encoding
```

也就是：

$$
\mathbf{z}_i=\mathbf{x}_i+\mathbf{p}_i
$$

其中：

- $\mathbf{x}_i$ 表示第 $i$ 个 token 的内容向量。
- $\mathbf{p}_i$ 表示第 $i$ 个位置的位置向量。
- $\mathbf{z}_i$ 表示送进 Transformer 的最终输入向量。

所以位置编码通常加在最开始：

```text
token -> token embedding
position -> position embedding / positional encoding
两者相加 -> 送入 Attention 层
```

## 6. 为什么是相加，而不是拼接

这是一个很常见的问题。

假设 token embedding 是：

```text
D 维
```

位置编码也设计成：

```text
D 维
```

这样两者就可以直接相加：

```text
D 维 + D 维 -> D 维
```

好处是输入维度不变。

如果拼接，就会变成：

```text
D 维 token 信息 + D 维位置信息 -> 2D 维
```

这会改变后面所有层的输入维度，设计上更麻烦。

相加的直觉是：

```text
在原来的 token 表示上，叠加一点位置信息。
```

相加后，模型仍然看到一个 D 维向量，只是这个向量同时包含内容和位置。

## 7. 用形状看位置编码

假设一批输入的 token embedding 形状是：

```text
B x N x D
```

其中：

- $B$ 表示 batch size。
- $N$ 表示 token 数量。
- $D$ 表示每个 token 的向量维度。

位置编码也要能对应每个位置。

如果不考虑 batch，位置编码可以是：

```text
N x D
```

每一行表示一个位置的编码：

```text
第 1 行：位置 1 的编码
第 2 行：位置 2 的编码
...
第 N 行：位置 N 的编码
```

加到 batch 输入上时，可以广播成：

```text
B x N x D
```

所以相加后形状还是：

```text
B x N x D
```

这点很重要：

```text
位置编码改变信息内容，不改变整体形状。
```

## 8. 一个小数字例子

假设一句话有 4 个 token，每个 token 用 3 维向量表示。

token embedding 是：

```text
X：4 x 3
```

位置编码也是：

```text
P：4 x 3
```

相加后：

```text
Z = X + P
Z：4 x 3
```

展开看就是：

$$
\mathbf{z}_1=\mathbf{x}_1+\mathbf{p}_1
$$

$$
\mathbf{z}_2=\mathbf{x}_2+\mathbf{p}_2
$$

$$
\mathbf{z}_3=\mathbf{x}_3+\mathbf{p}_3
$$

$$
\mathbf{z}_4=\mathbf{x}_4+\mathbf{p}_4
$$

每个位置都拿到自己的 token 信息和位置信息。

## 9. 位置编码应该满足什么要求

一个好的位置编码，至少应该让模型知道：

```text
不同位置不一样。
相近位置和远距离位置应该能区分。
模型能利用这些信息理解顺序关系。
```

比如第 1 个位置和第 2 个位置应该有不同的位置编码。

第 2 个位置和第 10 个位置也应该能区分。

而且模型最好还能学到一些相对关系，比如：

```text
某个词在另一个词前面。
某个词距离另一个词很近。
某个词距离另一个词比较远。
```

位置编码的各种设计，都是围绕这些目标展开的。

## 10. 第一类：可学习位置编码

最直观的做法是：给每个位置准备一个可学习向量。

比如最大长度是 512，模型维度是 768。

那么可以准备一张位置表：

```text
512 x 768
```

第 1 行表示位置 1。

第 2 行表示位置 2。

第 512 行表示位置 512。

这些位置向量和普通参数一样，会在训练中更新。

这种方式叫可学习位置编码，或者 learned positional embedding。

它的直觉非常简单：

```text
让模型自己学每个位置应该用什么向量表示。
```

## 11. 可学习位置编码的优缺点

可学习位置编码的优点是简单直接。

模型可以根据任务自己学习位置表示。

但它也有一个问题：

```text
训练时没见过的位置，模型不一定自然会处理。
```

比如训练时最大长度是 512。

位置表只有 512 行。

如果推理时突然输入长度 800，就会遇到麻烦。

当然实际模型有各种扩展方法，但入门时先记住：

```text
可学习位置编码简单，但通常受最大长度限制。
```

## 12. 第二类：正弦余弦位置编码

Transformer 原始论文里使用的是正弦余弦位置编码。

它不是训练出来的参数。

它是用固定公式算出来的。

公式通常写成：

$$
PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

$$
PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

第一次看到不用紧张。

先读符号：

- $pos$ 表示位置，比如第 0 个、第 1 个、第 2 个位置。
- $i$ 表示维度编号的一部分。
- $d_{model}$ 表示模型维度，也就是每个 token 向量有多少维。
- 偶数维用 sin。
- 奇数维用 cos。

这一节先不要求你背公式，只要理解它想做什么。

## 13. 正弦余弦编码的直觉

正弦和余弦函数会随着位置变化而周期性变化。

不同维度使用不同频率。

可以粗略理解成：

```text
有些维度变化得很快，适合区分近距离位置。
有些维度变化得很慢，适合表示更长范围的位置。
```

就像用很多把不同刻度的尺子同时测量位置。

短刻度尺对附近变化敏感。

长刻度尺可以覆盖更大范围。

多个频率组合起来，每个位置就会得到一个独特的位置向量。

这就是正弦余弦位置编码的核心直觉：

```text
用不同频率的波形，为每个位置生成一个可区分的向量。
```

## 14. 为什么要同时用 sin 和 cos

sin 和 cos 是一对相位不同的周期函数。

你不需要从三角函数深处钻进去。

先记住直觉：

```text
sin 和 cos 搭配，可以更稳定地表达位置变化。
```

同一个频率下，sin 和 cos 提供两个互补的维度。

位置往前移动时，这两个值会一起按规律变化。

这样模型更容易从这些变化里学习相对位置信息。

入门阶段先不用推导它的全部数学性质。

先知道：

```text
sin/cos 位置编码用固定规律给每个位置生成向量。
不同位置得到不同向量。
相邻位置之间的变化也有规律。
```

## 15. 可学习位置编码和正弦余弦位置编码对比

先做一个简单对比。

| 类型 | 怎么得到 | 是否训练更新 | 直觉 |
|---|---|---|---|
| 可学习位置编码 | 查一张位置参数表 | 是 | 每个位置的表示由模型自己学 |
| 正弦余弦位置编码 | 用固定公式计算 | 否 | 用不同频率的波形表示位置 |

两者都能给模型提供位置信息。

区别在于：

```text
可学习位置编码更像查表。
正弦余弦位置编码更像按公式生成。
```

现在很多模型会使用不同的位置编码变体。

但入门阶段先掌握这两类，就够理解主线。

## 16. 位置编码进入 Attention 后发生什么

位置编码不是单独拿去算 Attention。

它先和 token embedding 相加：

$$
Z=X+P
$$

然后用 $Z$ 去生成 Q、K、V：

$$
Q=ZW_Q
$$

$$
K=ZW_K
$$

$$
V=ZW_V
$$

这意味着 Q、K、V 里面都可以间接包含位置信息。

所以模型在计算注意力时，不仅能考虑 token 内容，还能考虑位置。

可以这样说：

```text
位置编码先混入输入表示。
然后 Attention 在带有位置信息的表示上计算关系。
```

## 17. 为什么位置编码不是 attention weight

这里容易混。

位置编码不是注意力权重。

注意力权重是：

```text
Q 和 K 计算分数，再经过 Softmax 得到的动态权重。
```

位置编码是：

```text
输入进入 Attention 前，额外加入的位置信息。
```

二者位置不同、作用不同。

可以对比：

```text
位置编码：告诉模型 token 在哪里。
注意力权重：告诉模型当前 token 应该关注谁。
```

位置编码影响后面注意力权重的计算，但它本身不是权重。

## 18. 绝对位置和相对位置的直觉

位置可以有两种理解方式。

第一种是绝对位置：

```text
这个 token 是第 1 个。
这个 token 是第 5 个。
这个 token 是第 20 个。
```

第二种是相对位置：

```text
这个 token 在另一个 token 前面 1 个位置。
这个 token 距离另一个 token 5 个位置。
```

原始 Transformer 的正弦余弦位置编码属于绝对位置编码的一种。

后来很多模型也会使用相对位置相关的方法。

这一节先不深入各种现代变体，只先建立区别：

```text
绝对位置：我在第几个。
相对位置：我和你隔多远、谁在谁前后。
```

## 19. 位置编码和 CNN 的位置感有什么不同

CNN 处理图像时，卷积核在局部窗口滑动。

它天然利用了局部邻域结构。

比如 3 x 3 卷积核默认就知道附近像素是一组局部区域。

Self-Attention 不一样。

它允许每个位置直接看所有位置。

这种全局关系建模很灵活，但也使得顺序或位置需要额外表达。

所以可以这样对比：

```text
CNN：结构里天然带有局部位置关系。
Self-Attention：关系建模很自由，但需要显式加入位置信息。
```

这不是谁优谁劣，而是结构特点不同。

## 20. 位置编码和 mask 不一样

前面 3Blue1Brown 复盘课里提过 GPT 的 mask。

mask 和位置编码也不是一回事。

位置编码解决的是：

```text
模型怎么知道 token 的顺序和位置。
```

mask 解决的是：

```text
哪些位置允许看，哪些位置不允许看。
```

例如 GPT 预测下一个 token 时，不允许当前位置看未来。

这需要 causal mask。

所以：

```text
位置编码：提供顺序信息。
mask：限制注意力可见范围。
```

它们经常一起出现，但作用不同。

## 21. 常见误解 1：位置编码就是位置编号

位置编码不是简单把位置编号 1、2、3 直接塞进去。

如果只给一个整数编号，和 token embedding 的高维向量很难自然融合。

所以位置通常也表示成一个向量。

比如位置 3 不是只写成：

```text
3
```

而是表示成：

```text
[0.14, 0.99, -0.20, ...]
```

这样它才能和 token embedding 在同一个向量空间里相加。

## 22. 常见误解 2：加了位置编码后 Attention 就只看附近

不是。

位置编码只是告诉模型位置信息。

它不会强制模型只看附近。

Self-Attention 仍然可以让每个位置看所有位置。

只是现在模型在判断关系时，能利用顺序信息。

比如它可以学到：

```text
某些关系更可能出现在相邻位置。
某些关系可能跨越很远。
某些任务里不能看未来。
```

但这些都不是位置编码单独强制出来的。

位置编码只是把“位置信息”交给模型。

## 23. 常见误解 3：位置编码只在文本里有用

位置编码不只用于文本。

只要模型处理的是一组位置，就可能需要位置信息。

比如图像 Transformer 会把图片切成多个 patch。

每个 patch 像一个 token。

这时模型也需要知道：

```text
这个 patch 在图片左上角？
还是在图片中间？
还是在右下角？
```

否则同样的 patch 内容，出现在不同位置可能意义不同。

所以位置编码的本质是：

```text
给一组无位置信息的向量补上它们在结构中的位置。
```

## 24. 本节小结

这一节先记住：

1. Self-Attention 本身主要计算内容之间的关系，不天然知道顺序。
2. 位置编码给 token embedding 补充位置信息。
3. 常见输入形式是 `token embedding + positional encoding`。
4. token embedding 和位置编码通常形状相同，都是 `N x D` 或可广播到 `B x N x D`。
5. 相加后形状不变，但信息多了位置成分。
6. 可学习位置编码是通过参数表学习每个位置的向量。
7. 正弦余弦位置编码是用固定公式生成不同频率的位置向量。
8. 位置编码不是注意力权重，也不是 mask。
9. 绝对位置强调“我在第几个”，相对位置强调“我和你隔多远”。
10. 文本和图像 patch 等场景都可能需要位置信息。

## 25. 自测问题

1. 为什么 Self-Attention 本身不天然知道 token 顺序？
2. 为什么“我喜欢你”和“你喜欢我”能说明顺序很重要？
3. 位置编码可以先理解成给 token 发什么？
4. token embedding 和 positional encoding 分别表示什么信息？
5. 为什么位置编码通常和 token embedding 相加？
6. 如果 token embedding 是 `B x N x D`，位置编码通常需要什么形状？
7. 可学习位置编码是什么？
8. 正弦余弦位置编码为什么使用不同频率？
9. 位置编码和注意力权重有什么区别？
10. 位置编码和 mask 有什么区别？
11. 什么是绝对位置？什么是相对位置？
12. 为什么图像 Transformer 也可能需要位置编码？